# Notebook 02 - Análise STRIDE

**Tech Challenge - Fase 5 - Hackaton FIAP**
**Modelagem de Ameaças utilizando IA (STRIDE)**

Este notebook executa os Passos 3 e 4 do pipeline:
- **Passo 3**: Análise STRIDE por componente via Claude API
- **Passo 4**: Geração de relatórios (JSON estruturado + HTML formatado)

**GPU necessária**: Nenhuma (apenas chamadas API)

> **Configuração necessária antes de executar:**
> 1. No menu lateral do Colab, clique em 🔑 **Secrets** e adicione:
>    - `ANTHROPIC_API_KEY`: sua chave de API da Anthropic (obtenha em https://console.anthropic.com)
> 2. No menu **Ambiente de execução > Alterar tipo de ambiente de execução**, selecione:
>    - **Versão do ambiente de execução**: `2025.07`
>    - **CPU** (suficiente — este notebook não usa GPU)
>    - Também funciona com T4 ou A100, mas não é necessário

## 1. Setup e Dependências

In [ ]:
# Instalar dependências
!pip install -q anthropic

In [ ]:
import os
import json
import time
import base64
from pathlib import Path
from datetime import datetime

import anthropic

print("Dependências carregadas com sucesso!")

## 2. Montar Google Drive e Configurar

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# Caminhos
DRIVE_BASE = Path('/content/drive/MyDrive/hackaton-stride')
DETECTIONS_DIR = DRIVE_BASE / 'outputs' / 'detections'
REPORTS_DIR = DRIVE_BASE / 'outputs' / 'reports'
IMAGES_DIR = DRIVE_BASE / 'test_images'

# Criar todos os diretórios necessários
for d in [IMAGES_DIR, DETECTIONS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Baixar imagens de teste do GitHub (se não existirem no Drive)
GITHUB_RAW = 'https://raw.githubusercontent.com/gledson85/Tech-Challenge---Fase-5-Hackaton/main/imagens'
for idx in [1, 2]:
    img_path = IMAGES_DIR / f'arquitetura {idx}.png'
    if not img_path.exists():
        url = f'{GITHUB_RAW}/arquitetura%20{idx}.png'
        print(f"Baixando arquitetura {idx}.png do GitHub...")
        !wget -q -O "{img_path}" "{url}"
        print(f"  Salvo em: {img_path}")
    else:
        print(f"arquitetura {idx}.png já existe no Drive")

# API Claude
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print(f"\nDetecções disponíveis: {list(DETECTIONS_DIR.glob('*.json'))}")

In [ ]:
# Selecionar arquitetura (1 = AWS, 2 = Azure)
ARCH_INDEX = 1

# Carregar JSON de componentes detectados
detections_file = DETECTIONS_DIR / f'componentes_arquitetura_{ARCH_INDEX}.json'
with open(detections_file, 'r', encoding='utf-8') as f:
    detection_data = json.load(f)

provider = detection_data['provider']
components = detection_data['components']
topology = detection_data.get('topology', {'groups': [], 'connections': [], 'data_flow': []})

print(f"Arquitetura: {ARCH_INDEX} ({provider})")
print(f"Componentes carregados: {len(components)}")
for c in components:
    print(f"  - {c['name']} ({c['class']})")

print(f"\nTopologia:")
print(f"  Grupos: {len(topology.get('groups', []))}")
print(f"  Conexões: {len(topology.get('connections', []))}")
print(f"  Fluxo: {' -> '.join(topology.get('data_flow', []))}")

## 3. Passo 3 — Análise STRIDE por Componente

In [ ]:
import re

def clean_json_text(text: str) -> str:
    """
    Limpa texto retornado pelo Claude para extrair JSON válido.
    Remove code fences, trailing commas e outros artefatos.
    """
    text = re.sub(r'```(?:json)?\s*\n?', '', text).strip()
    text = re.sub(r',\s*([}\]])', r'\1', text)
    return text


def parse_stride_json(text: str) -> dict:
    """
    Parser robusto para JSON STRIDE retornado pelo Claude.
    Tenta múltiplas estratégias de parsing.
    """
    # Estratégia 1: limpeza padrão + parse completo
    try:
        cleaned = clean_json_text(text)
        start = cleaned.find('{')
        end = cleaned.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(cleaned[start:end])
    except json.JSONDecodeError:
        pass

    # Estratégia 2: extrair bloco bruto entre { }
    try:
        raw = text[text.find('{'):text.rfind('}') + 1]
        raw = re.sub(r',\s*([}\]])', r'\1', raw)
        return json.loads(raw)
    except (json.JSONDecodeError, ValueError):
        pass

    # Estratégia 3: extrair cada categoria STRIDE individualmente via regex
    categories = [
        'spoofing', 'tampering', 'repudiation',
        'information_disclosure', 'denial_of_service', 'elevation_of_privilege'
    ]
    result = {}
    for cat in categories:
        # Buscar o bloco de cada categoria: "cat": { ... }
        pattern = rf'"{cat}"\s*:\s*\{{([^}}]*)\}}'
        match = re.search(pattern, text, re.DOTALL)
        if match:
            try:
                inner = match.group(1)
                # Extrair campos individualmente
                threat = re.search(r'"threat"\s*:\s*"((?:[^"\\]|\\.)*)"', inner)
                risk = re.search(r'"risk"\s*:\s*"((?:[^"\\]|\\.)*)"', inner)
                counter = re.search(r'"countermeasure"\s*:\s*"((?:[^"\\]|\\.)*)"', inner)
                ref = re.search(r'"reference"\s*:\s*"((?:[^"\\]|\\.)*)"', inner)
                result[cat] = {
                    'threat': threat.group(1) if threat else 'N/A',
                    'risk': risk.group(1) if risk else 'Médio',
                    'countermeasure': counter.group(1) if counter else 'N/A',
                    'reference': ref.group(1) if ref else 'N/A',
                }
            except Exception:
                pass

    return result if result else {}


def get_component_topology_context(component: dict, topology: dict) -> str:
    """
    Retorna contexto topológico de um componente:
    em quais grupos está, com quem se conecta.
    """
    lines = []
    comp_name = component['name']

    # Grupos que contêm este componente
    for g in topology.get('groups', []):
        if comp_name in g.get('contains', []):
            parent = f" (dentro de {g['parent']})" if g.get('parent') else ""
            lines.append(f"  - Está no grupo: {g['name']} [{g.get('type', '?')}]{parent}")

    # Conexões deste componente
    for conn in topology.get('connections', []):
        if conn['from'] == comp_name:
            lines.append(f"  - Conecta para: {conn['to']} ({conn.get('description', '')})")
        elif conn['to'] == comp_name:
            lines.append(f"  - Recebe de: {conn['from']} ({conn.get('description', '')})")

    # Posição no fluxo de dados
    data_flow = topology.get('data_flow', [])
    if comp_name in data_flow:
        idx = data_flow.index(comp_name)
        lines.append(f"  - Posição no fluxo de dados: {idx + 1}/{len(data_flow)}")

    return "\n".join(lines) if lines else "  (sem informação topológica disponível)"


def build_stride_prompt(component: dict, all_components: list, provider: str,
                        topology: dict) -> str:
    """
    Constrói o prompt STRIDE contextualizado para um componente.
    Inclui contexto da arquitetura completa e topologia.
    """
    arch_context = "\n".join(
        f"  - {c['name']} ({c['class']})"
        for c in all_components
    )

    topo_context = get_component_topology_context(component, topology)

    # Fluxo de dados geral
    data_flow = topology.get('data_flow', [])
    flow_str = " -> ".join(data_flow) if data_flow else "(não disponível)"

    prompt = f"""Você é um especialista em segurança de software e modelagem de ameaças.

Analise o componente abaixo usando a metodologia STRIDE. O componente faz parte de uma arquitetura {provider}.

## Componente a analisar
- Nome: {component['name']}
- Classe: {component['class']}
- Provedor: {provider}

## Posição na topologia
{topo_context}

## Fluxo de dados da arquitetura
{flow_str}

## Contexto da arquitetura completa
Outros componentes presentes:
{arch_context}

## Instruções

Para cada categoria STRIDE, forneça:
1. Descrição da ameaça específica para este componente (máximo 150 caracteres)
2. Nível de risco: "Alto", "Médio" ou "Baixo"
3. Contramedida específica do provedor ({provider}) (máximo 150 caracteres)
4. Referência a boas práticas ({"AWS Well-Architected Framework" if provider == "AWS" else "Azure Security Benchmark"}) (máximo 100 caracteres)

IMPORTANTE: Responda APENAS com JSON puro, sem code fences (```), sem comentários, sem texto antes ou depois. Não use aspas duplas dentro dos valores de texto.
{{"spoofing": {{"threat": "...", "risk": "Alto|Médio|Baixo", "countermeasure": "...", "reference": "..."}},
"tampering": {{"threat": "...", "risk": "...", "countermeasure": "...", "reference": "..."}},
"repudiation": {{"threat": "...", "risk": "...", "countermeasure": "...", "reference": "..."}},
"information_disclosure": {{"threat": "...", "risk": "...", "countermeasure": "...", "reference": "..."}},
"denial_of_service": {{"threat": "...", "risk": "...", "countermeasure": "...", "reference": "..."}},
"elevation_of_privilege": {{"threat": "...", "risk": "...", "countermeasure": "...", "reference": "..."}}}}"""

    return prompt


def analyze_stride(component: dict, all_components: list, provider: str,
                   topology: dict) -> dict:
    """
    Executa análise STRIDE para um componente usando Claude API.
    """
    prompt = build_stride_prompt(component, all_components, provider, topology)

    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=2000,
        messages=[
            {'role': 'user', 'content': prompt}
        ]
    )

    text = response.content[0].text.strip()
    result = parse_stride_json(text)

    if not result:
        print(f"  Erro: não foi possível parsear JSON para {component['name']}")

    return result

In [ ]:
# Executar análise STRIDE para todos os componentes
print(f"Iniciando análise STRIDE para {len(components)} componentes...\n")

stride_results = []
t0 = time.time()

for idx, component in enumerate(components):
    print(f"[{idx+1}/{len(components)}] Analisando: {component['name']} ({component['class']})...")

    threats = analyze_stride(component, components, provider, topology)

    result = {
        'id': component['id'],
        'name': component['name'],
        'class': component['class'],
        'bbox': component['bbox'],
        'provider': provider,
        'threats': threats
    }
    stride_results.append(result)

    # Resumo rápido
    risks = [threats[cat].get('risk', '?') for cat in threats if isinstance(threats[cat], dict)]
    high_count = risks.count('Alto')
    print(f"  -> {len(threats)} categorias analisadas, {high_count} riscos altos")

    # Rate limiting
    time.sleep(1)

t_stride = time.time() - t0
print(f"\nAnálise STRIDE concluída em {t_stride:.2f}s")

## 4. Passo 4a — Gerar Relatório JSON

In [ ]:
def generate_summary(stride_results: list) -> dict:
    """
    Gera resumo executivo da análise STRIDE.
    """
    total_threats = 0
    risk_counts = {'Alto': 0, 'Médio': 0, 'Baixo': 0}
    category_counts = {
        'spoofing': 0, 'tampering': 0, 'repudiation': 0,
        'information_disclosure': 0, 'denial_of_service': 0,
        'elevation_of_privilege': 0
    }
    high_risk_components = []

    for result in stride_results:
        threats = result.get('threats', {})
        component_high = 0
        for cat, details in threats.items():
            if isinstance(details, dict):
                total_threats += 1
                risk = details.get('risk', 'Baixo')
                if risk in risk_counts:
                    risk_counts[risk] += 1
                if risk == 'Alto':
                    component_high += 1
                if cat in category_counts:
                    category_counts[cat] += 1

        if component_high > 0:
            high_risk_components.append({
                'name': result['name'],
                'class': result['class'],
                'high_risk_count': component_high
            })

    return {
        'total_components': len(stride_results),
        'total_threats': total_threats,
        'risk_distribution': risk_counts,
        'threats_by_category': category_counts,
        'high_risk_components': sorted(
            high_risk_components,
            key=lambda x: x['high_risk_count'],
            reverse=True
        )
    }

In [ ]:
# Gerar resumo e salvar JSON completo
summary = generate_summary(stride_results)

report_data = {
    'metadata': {
        'project': 'Tech Challenge - Fase 5 - Hackaton FIAP',
        'methodology': 'STRIDE',
        'generated_at': datetime.now().isoformat(),
        'image': detection_data['image'],
        'provider': provider
    },
    'components': stride_results,
    'summary': summary,
    'timing': {
        'stride_analysis_s': round(t_stride, 2),
        'detection_timing': detection_data.get('timing', {})
    }
}

# Salvar JSON
json_path = REPORTS_DIR / f'stride_arquitetura_{ARCH_INDEX}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2)

print(f"Relatório JSON salvo em: {json_path}")
print(f"\nResumo executivo:")
print(f"  Componentes analisados: {summary['total_components']}")
print(f"  Total de ameaças: {summary['total_threats']}")
print(f"  Riscos: Alto={summary['risk_distribution']['Alto']}, "
      f"Médio={summary['risk_distribution']['Médio']}, "
      f"Baixo={summary['risk_distribution']['Baixo']}")
if summary['high_risk_components']:
    print(f"\n  Componentes com risco alto:")
    for c in summary['high_risk_components']:
        print(f"    - {c['name']}: {c['high_risk_count']} ameaças de alto risco")

## 5. Passo 4b — Gerar Relatório HTML

In [ ]:
def generate_html_report(stride_results: list, summary: dict, provider: str,
                         arch_index: int, topology: dict = None,
                         original_image_b64: str = None,
                         annotated_image_b64: str = None) -> str:
    """Gera relatório HTML completo da análise STRIDE com topologia e imagens."""

    category_names = {
        'spoofing': 'S - Spoofing (Falsificação)',
        'tampering': 'T - Tampering (Adulteração)',
        'repudiation': 'R - Repudiation (Negação)',
        'information_disclosure': 'I - Information Disclosure (Vazamento)',
        'denial_of_service': 'D - Denial of Service (Negação de Serviço)',
        'elevation_of_privilege': 'E - Elevation of Privilege (Elevação)'
    }

    risk_colors = {
        'Alto': ('#dc3545', '#fff'),
        'Médio': ('#ffc107', '#333'),
        'Baixo': ('#28a745', '#fff'),
    }

    generated_at = datetime.now().strftime('%d/%m/%Y %H:%M')
    if topology is None:
        topology = {'groups': [], 'connections': [], 'data_flow': []}

    # --- Mapa de numeração: nome do componente → número sequencial ---
    comp_number = {r['name']: i + 1 for i, r in enumerate(stride_results)}

    # --- Mapa de localização topológica: nome do componente → grupo mais específico ---
    def get_component_location(comp_name):
        """Retorna o grupo mais específico (menor) que contém o componente."""
        locations = []
        for g in topology.get('groups', []):
            if comp_name in g.get('contains', []):
                locations.append(g['name'])
        # Retorna o último encontrado (geralmente o mais específico/filho)
        return locations[-1] if locations else ''

    comp_location = {r['name']: get_component_location(r['name']) for r in stride_results}

    # Detectar nomes duplicados para marcar com localização
    from collections import Counter
    name_counts = Counter(r['name'] for r in stride_results)
    has_duplicates = {name for name, count in name_counts.items() if count > 1}

    # --- Seção de imagens da arquitetura ---
    images_html = ''
    if original_image_b64 or annotated_image_b64:
        images_html = '<h1 style="margin-top:40px">Diagrama da Arquitetura</h1>'
        if original_image_b64:
            images_html += '''
            <h3>Imagem Original</h3>
            <div style="text-align:center;margin-bottom:20px">
                <img src="data:image/png;base64,''' + original_image_b64 + '''" style="max-width:100%;border:1px solid #ddd;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1)" alt="Diagrama original">
            </div>'''
        if annotated_image_b64:
            images_html += '''
            <h3>Componentes Detectados</h3>
            <div style="text-align:center;margin-bottom:20px">
                <img src="data:image/png;base64,''' + annotated_image_b64 + '''" style="max-width:100%;border:1px solid #ddd;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1)" alt="Componentes detectados">
            </div>'''

    # --- Componentes table rows (com localização) ---
    comp_rows = ''
    for i, r in enumerate(stride_results):
        bg = '#f9f9f9' if i % 2 == 0 else '#fff'
        loc = comp_location.get(r['name'], '')
        loc_html = f'<span style="color:#666;font-size:12px">{loc}</span>' if loc else '<span style="color:#ccc;font-size:12px">—</span>'
        comp_rows += f'''<tr style="background:{bg}">
            <td style="padding:6px 10px;border:1px solid #ddd;text-align:center">{i+1}</td>
            <td style="padding:6px 10px;border:1px solid #ddd">{r['name']}</td>
            <td style="padding:6px 10px;border:1px solid #ddd;text-align:center">{r['class']}</td>
            <td style="padding:6px 10px;border:1px solid #ddd">{loc_html}</td>
            <td style="padding:6px 10px;border:1px solid #ddd;text-align:center">{r['provider']}</td>
        </tr>'''

    # --- High risk components (com numeração) ---
    high_risk_html = ''
    if summary.get('high_risk_components'):
        items = ''
        for c in summary['high_risk_components']:
            num = comp_number.get(c['name'], '?')
            loc = comp_location.get(c['name'], '')
            loc_text = f' <span style="color:#999;font-size:12px">({loc})</span>' if loc else ''
            items += f"<li><strong>#{num} {c['name']}</strong>{loc_text}: {c['high_risk_count']} ameaças de alto risco</li>"
        high_risk_html = f'''
        <h3 style="color:#dc3545;margin-top:20px">Componentes com Risco Alto</h3>
        <ul>{items}</ul>'''

    # --- Seção de Topologia ---
    topology_html = ''
    data_flow = topology.get('data_flow', [])
    groups = topology.get('groups', [])
    connections = topology.get('connections', [])

    if data_flow or groups or connections:
        topology_html = '<h1 style="margin-top:40px">Topologia da Arquitetura</h1>'

        # Fluxo de dados
        if data_flow:
            flow_steps = ''
            for i, step in enumerate(data_flow):
                num = comp_number.get(step, '?')
                flow_steps += f'<span style="display:inline-block;background:#e3f2fd;border:1px solid #90caf9;padding:4px 12px;border-radius:6px;font-size:13px;font-weight:bold">#{num} {step}</span>'
                if i < len(data_flow) - 1:
                    flow_steps += '<span style="display:inline-block;padding:0 6px;color:#90caf9;font-size:18px;vertical-align:middle">&#10140;</span>'
            topology_html += f'''
            <h3>Fluxo de Dados</h3>
            <div style="padding:15px;background:#fafafa;border:1px solid #e0e0e0;border-radius:8px;line-height:2.2;margin-bottom:20px">
                {flow_steps}
            </div>'''

        # Grupos/containers
        if groups:
            groups_html = ''
            for g in groups:
                parent_info = f' <span style="color:#999;font-size:12px">(dentro de {g["parent"]})</span>' if g.get('parent') else ''
                contains = g.get('contains', [])
                if contains:
                    chips = ' '.join(
                        f'<span style="display:inline-block;background:#fff;border:1px solid #ccc;padding:2px 8px;border-radius:4px;font-size:12px;margin:2px">#{comp_number.get(c, "?")} {c}</span>'
                        for c in contains
                    )
                else:
                    chips = '<span style="color:#999;font-size:12px">vazio</span>'
                groups_html += f'''
                <div style="margin-bottom:10px;padding:10px 14px;background:#f5f5f5;border-left:4px solid #1565c0;border-radius:4px">
                    <strong>{g['name']}</strong> <span style="color:#666;font-size:12px">[{g.get('type', '?')}]</span>{parent_info}
                    <div style="margin-top:6px">{chips}</div>
                </div>'''
            topology_html += f'<h3>Grupos e Containers</h3>{groups_html}'

        # Conexões
        if connections:
            conn_rows = ''
            for c in connections:
                from_num = comp_number.get(c['from'], '?')
                to_num = comp_number.get(c['to'], '?')
                conn_rows += f'''<tr>
                    <td style="padding:5px 10px;border:1px solid #ddd">#{from_num} {c['from']}</td>
                    <td style="padding:5px 10px;border:1px solid #ddd;text-align:center">&#10140;</td>
                    <td style="padding:5px 10px;border:1px solid #ddd">#{to_num} {c['to']}</td>
                    <td style="padding:5px 10px;border:1px solid #ddd;color:#666;font-size:12px">{c.get('description', '')}</td>
                </tr>'''
            topology_html += f'''
            <h3 style="margin-top:20px">Conexões</h3>
            <table>
                <thead><tr>
                    <th>Origem</th><th style="width:30px"></th><th>Destino</th><th>Descrição</th>
                </tr></thead>
                <tbody>{conn_rows}</tbody>
            </table>'''

    # --- Nota sobre componentes repetidos ---
    duplicate_note = ''
    if has_duplicates:
        duplicate_note = '''
        <div style="margin:20px 0;padding:12px 16px;background:#fff3cd;border:1px solid #ffc107;border-radius:6px;font-size:13px;color:#856404">
            <strong>Nota:</strong> Componentes com o mesmo nome aparecem mais de uma vez quando estão em diferentes zonas de disponibilidade ou grupos.
            A análise STRIDE de cada instância considera sua posição específica na topologia, conexões e fluxo de dados, podendo resultar em riscos e contramedidas distintas.
        </div>'''

    # --- STRIDE details per component (com numeração e localização) ---
    stride_details = ''
    for i, result in enumerate(stride_results):
        num = i + 1
        threats = result.get('threats', {})
        loc = comp_location.get(result['name'], '')
        loc_badge = f' <span style="background:#e3f2fd;color:#1565c0;padding:2px 10px;border-radius:10px;font-size:12px;font-weight:normal">{loc}</span>' if loc else ''

        # Conexões deste componente (contexto)
        comp_conns = []
        for conn in topology.get('connections', []):
            if conn.get('from') == result['name']:
                comp_conns.append(f"→ {conn['to']}")
            elif conn.get('to') == result['name']:
                comp_conns.append(f"← {conn['from']}")
        conns_html = ''
        if comp_conns:
            conns_chips = ' '.join(
                f'<span style="display:inline-block;background:#f5f5f5;border:1px solid #ddd;padding:1px 8px;border-radius:4px;font-size:11px;margin:1px">{c}</span>'
                for c in comp_conns[:6]  # máximo 6 conexões visíveis
            )
            if len(comp_conns) > 6:
                conns_chips += f' <span style="color:#999;font-size:11px">+{len(comp_conns)-6} mais</span>'
            conns_html = f'<div style="margin-top:4px;margin-bottom:10px"><strong style="font-size:12px;color:#666">Conexões:</strong> {conns_chips}</div>'

        cats_html = ''
        for cat_key, cat_name in category_names.items():
            if cat_key not in threats or not isinstance(threats[cat_key], dict):
                continue
            d = threats[cat_key]
            risk = d.get('risk', 'Baixo')
            bg_color, fg_color = risk_colors.get(risk, ('#6c757d', '#fff'))
            cats_html += f'''
            <div style="margin-bottom:12px;padding:10px;background:#f8f9fa;border-left:4px solid {bg_color};border-radius:4px">
                <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px">
                    <strong style="font-size:14px">{cat_name}</strong>
                    <span style="background:{bg_color};color:{fg_color};padding:2px 10px;border-radius:12px;font-size:12px;font-weight:bold">{risk}</span>
                </div>
                <div style="font-size:13px;line-height:1.5">
                    <div><strong>Ameaça:</strong> {d.get('threat', 'N/A')}</div>
                    <div><strong>Contramedida:</strong> {d.get('countermeasure', 'N/A')}</div>
                    <div style="color:#666;font-size:12px"><strong>Ref:</strong> {d.get('reference', 'N/A')}</div>
                </div>
            </div>'''

        stride_details += f'''
        <div style="page-break-before:always;margin-top:30px">
            <h2 style="color:#333;border-bottom:2px solid #007bff;padding-bottom:5px">#{num} {result['name']}{loc_badge}</h2>
            <p style="color:#666;margin-bottom:5px">Classe: {result['class']} | Provedor: {result['provider']}</p>
            {conns_html}
            {cats_html}
        </div>'''

    # --- Risk distribution badges ---
    risk_badges = ''
    for level in ['Alto', 'Médio', 'Baixo']:
        count = summary['risk_distribution'].get(level, 0)
        bg_color, fg_color = risk_colors.get(level, ('#6c757d', '#fff'))
        risk_badges += f'<span style="display:inline-block;background:{bg_color};color:{fg_color};padding:4px 14px;border-radius:14px;margin-right:10px;font-weight:bold">{level}: {count}</span>'

    html = f'''<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Relatório STRIDE - Arquitetura {arch_index} ({provider})</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; margin: 0; padding: 0; color: #333; line-height: 1.6; }}
        .cover {{ text-align: center; padding: 50px 20px; background: linear-gradient(135deg, #2196F3 0%, #1976D2 50%, #1565C0 100%); color: white; }}
        .cover h1, .cover h2, .cover p {{ color: white; }}
        .cover h1 {{ font-size: 36px; margin-bottom: 5px; }}
        .cover h2 {{ font-size: 24px; font-weight: normal; opacity: 0.9; }}
        .cover p {{ font-size: 14px; opacity: 0.8; margin-top: 30px; }}
        .content {{ max-width: 900px; margin: 0 auto; padding: 30px 20px; }}
        h1 {{ color: #1565C0; }}
        table {{ width: 100%; border-collapse: collapse; margin: 15px 0; }}
        th {{ background: #1976D2; color: white; padding: 8px 10px; text-align: center; border: 1px solid #ddd; }}
        @media print {{ .cover {{ page-break-after: always; }} div[style*="page-break-before"] {{ page-break-before: always; }} }}
    </style>
</head>
<body>
    <div class="cover">
        <h1>Relatório de Modelagem<br>de Ameaças - STRIDE</h1>
        <h2>Arquitetura {arch_index} - {provider}</h2>
        <p>Tech Challenge - Fase 5 - Hackaton<br>FIAP - Pós Tech - Software Security<br><br>Gerado em: {generated_at}</p>
    </div>

    <div class="content">
        <h1>Resumo Executivo</h1>

        <h3>Visão Geral</h3>
        <p>Componentes analisados: <strong>{summary['total_components']}</strong><br>
        Total de ameaças identificadas: <strong>{summary['total_threats']}</strong></p>

        <h3>Distribuição de Riscos</h3>
        <p>{risk_badges}</p>

        {high_risk_html}

        {images_html}

        {topology_html}

        <h1 style="margin-top:40px">Componentes Detectados</h1>
        <table>
            <thead>
                <tr>
                    <th style="width:40px">#</th>
                    <th>Componente</th>
                    <th>Classe</th>
                    <th>Localização</th>
                    <th>Provedor</th>
                </tr>
            </thead>
            <tbody>
                {comp_rows}
            </tbody>
        </table>

        {duplicate_note}

        <h1 style="margin-top:40px">Análise STRIDE Detalhada</h1>
        {stride_details}
    </div>
</body>
</html>'''

    return html

In [ ]:
# Gerar HTML com imagens embutidas em base64
print("Gerando relatório HTML...")

# Codificar imagem original
original_image_path = IMAGES_DIR / f'arquitetura {ARCH_INDEX}.png'
original_b64 = None
if original_image_path.exists():
    with open(original_image_path, 'rb') as f:
        original_b64 = base64.b64encode(f.read()).decode('utf-8')
    print(f"  Imagem original: {original_image_path.name}")

# Codificar imagem anotada (gerada pelo nb01)
annotated_image_path = DETECTIONS_DIR / f'annotated_arquitetura_{ARCH_INDEX}.png'
annotated_b64 = None
if annotated_image_path.exists():
    with open(annotated_image_path, 'rb') as f:
        annotated_b64 = base64.b64encode(f.read()).decode('utf-8')
    print(f"  Imagem anotada: {annotated_image_path.name}")
else:
    print(f"  Imagem anotada não encontrada (execute o nb01 primeiro)")

html_content = generate_html_report(
    stride_results, summary, provider, ARCH_INDEX, topology,
    original_image_b64=original_b64,
    annotated_image_b64=annotated_b64
)

html_path = REPORTS_DIR / f'stride_arquitetura_{ARCH_INDEX}.html'
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\nRelatório HTML salvo em: {html_path}")
print(f"Abra o arquivo no navegador para visualizar.")
print(f"Para salvar como PDF, use Ctrl+P > Salvar como PDF.")

## 6. Resumo da Execução

In [ ]:
print("=" * 60)
print(f"RESUMO - Análise STRIDE")
print("=" * 60)
print(f"Arquitetura: {ARCH_INDEX} ({provider})")
print(f"Componentes analisados: {summary['total_components']}")
print(f"Total de ameaças: {summary['total_threats']}")
print(f"")
print(f"Distribuição de riscos:")
print(f"  Alto:  {summary['risk_distribution']['Alto']}")
print(f"  Médio: {summary['risk_distribution']['Médio']}")
print(f"  Baixo: {summary['risk_distribution']['Baixo']}")
print(f"")
print(f"Ameaças por categoria:")
for cat, count in summary['threats_by_category'].items():
    print(f"  {cat}: {count}")
print(f"")
print(f"Tempo análise STRIDE: {t_stride:.2f}s")
print(f"")
print(f"Arquivos gerados:")
print(f"  JSON: {json_path}")
print(f"  HTML: {html_path}")
print("=" * 60)

## 7. Estimativa de Custo

In [ ]:
# Estimativa de custo do Notebook 02
# Preços Claude Sonnet 4.6: $3.00/1M tokens input, $15.00/1M tokens output
# Cotação: US$ 1.00 = R$ 5.50

PRICE_INPUT = 3.00 / 1_000_000
PRICE_OUTPUT = 15.00 / 1_000_000
USD_BRL = 5.50

n_components = len(components)

# Passo 3: Análise STRIDE (N chamadas, prompt longo com topologia)
step3_input = n_components * 3_500   # ~3.5k tokens por chamada (prompt + topologia + contexto)
step3_output = n_components * 5_00   # ~500 tokens de output por chamada (JSON STRIDE)
step3_cost = step3_input * PRICE_INPUT + step3_output * PRICE_OUTPUT

# Passo 4: Geração de relatório (local, sem custo API)
# HTML gerado localmente (sem dependências externas)

# Este notebook não usa GPU (sem custo Colab adicional significativo)
total_cost = step3_cost

print("=" * 60)
print(f"ESTIMATIVA DE CUSTO - Notebook 02")
print("=" * 60)
print(f"")
print(f"Claude API (Sonnet 4.6):")
print(f"  Passo 3 - Análise STRIDE: ${step3_cost:.4f} (R$ {step3_cost * USD_BRL:.4f})")
print(f"    ({n_components} chamadas, ~3.5k input + ~500 output tokens cada)")
print(f"")
print(f"Passo 4 - Relatório: gerado localmente em HTML (sem custo API)")
print(f"Google Colab: sem GPU necessária (custo desprezível)")
print(f"")
print(f"{'─' * 60}")
print(f"TOTAL ESTIMADO:             ${total_cost:.4f} (R$ {total_cost * USD_BRL:.4f})")
print(f"{'─' * 60}")
print(f"")
print(f"Nota: Valores aproximados. Cotação: US$ 1.00 = R$ {USD_BRL:.2f}")